# Library Management System (Python + MySQL)

A notebook version of the Library Management System project.
Run the cells in order:
1. Install dependencies
2. Set up the database connection
3. Define the core functions
4. Use the functions interactively at the bottom

Make sure you've already run `schema.sql` in MySQL Workbench to create the `library_db` database and tables.

## 1. Install dependencies

In [1]:
import sys
!{sys.executable} -m pip install mysql-connector-python

## 2. Database connection

Update `DB_CONFIG` below with your own MySQL credentials.

In [2]:
import mysql.connector
from mysql.connector import Error
from datetime import date

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "Safal123@",   # <-- change this
    "database": "library_db"
}

def get_connection():
    """Return a new MySQL connection using the configured credentials."""
    try:
        connection = mysql.connector.connect(**DB_CONFIG)
        return connection
    except Error as e:
        print(f"Error connecting to MySQL: {e}")
        return None

# Quick connection test
test_conn = get_connection()
if test_conn and test_conn.is_connected():
    print("Connected to MySQL successfully.")
    test_conn.close()


Connected to MySQL successfully.


## 3. Core functions (books, members, transactions)

In [3]:
def add_book(title, author, genre, copies):
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute(
        """INSERT INTO books (title, author, genre, total_copies, available_copies)
           VALUES (%s, %s, %s, %s, %s)""",
        (title, author, genre, copies, copies)
    )
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Book '{title}' added successfully.")


def view_books():
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute("SELECT book_id, title, author, genre, available_copies, total_copies FROM books")
    rows = cursor.fetchall()
    cursor.close()
    conn.close()

    if not rows:
        print("No books found.")
        return

    print(f"{'ID':<5}{'Title':<30}{'Author':<20}{'Genre':<15}{'Available':<10}{'Total':<5}")
    for row in rows:
        print(f"{row[0]:<5}{row[1]:<30}{row[2]:<20}{row[3]:<15}{row[4]:<10}{row[5]:<5}")


In [7]:
def add_member(name, email, phone):
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO members (name, email, phone) VALUES (%s, %s, %s)",
        (name, email, phone)
    )
    conn.commit()
    cursor.close()
    conn.close()
    print(f"Member '{name}' added successfully.")


def view_members():
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute("SELECT member_id, name, email, phone FROM members")
    rows = cursor.fetchall()
    cursor.close()
    conn.close()

    if not rows:
        print("No members found.")
        return

    print(f"{'ID':<5}{'Name':<25}{'Email':<30}{'Phone':<15}")
    for row in rows:
        print(f"{row[0]:<5}{row[1]:<25}{row[2]:<30}{row[3]:<15}")


In [8]:
def issue_book(book_id, member_id):
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute("SELECT available_copies FROM books WHERE book_id = %s", (book_id,))
    result = cursor.fetchone()

    if not result:
        print("Book ID not found.")
    elif result[0] <= 0:
        print("No copies available for this book.")
    else:
        cursor.execute(
            """INSERT INTO transactions (book_id, member_id, issue_date, status)
               VALUES (%s, %s, %s, 'issued')""",
            (book_id, member_id, date.today())
        )
        cursor.execute(
            "UPDATE books SET available_copies = available_copies - 1 WHERE book_id = %s",
            (book_id,)
        )
        conn.commit()
        print("Book issued successfully.")

    cursor.close()
    conn.close()


def return_book(transaction_id, book_id):
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE transactions SET return_date = %s, status = 'returned' WHERE transaction_id = %s",
        (date.today(), transaction_id)
    )
    cursor.execute(
        "UPDATE books SET available_copies = available_copies + 1 WHERE book_id = %s",
        (book_id,)
    )
    conn.commit()
    cursor.close()
    conn.close()
    print("Book returned successfully.")


In [9]:
def view_transactions():
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute(
        """SELECT t.transaction_id, b.title, m.name, t.issue_date, t.return_date, t.status
           FROM transactions t
           JOIN books b ON t.book_id = b.book_id
           JOIN members m ON t.member_id = m.member_id
           ORDER BY t.transaction_id DESC"""
    )
    rows = cursor.fetchall()
    cursor.close()
    conn.close()

    if not rows:
        print("No transactions found.")
        return

    print(f"{'ID':<5}{'Book':<25}{'Member':<20}{'Issued':<12}{'Returned':<12}{'Status':<10}")
    for row in rows:
        returned = row[4] if row[4] else "-"
        print(f"{row[0]:<5}{row[1]:<25}{row[2]:<20}{str(row[3]):<12}{str(returned):<12}{row[5]:<10}")


def most_borrowed_books():
    conn = get_connection()
    if not conn:
        return
    cursor = conn.cursor()
    cursor.execute(
        """SELECT b.title, COUNT(*) AS times_borrowed
           FROM transactions t
           JOIN books b ON t.book_id = b.book_id
           GROUP BY b.title
           ORDER BY times_borrowed DESC"""
    )
    rows = cursor.fetchall()
    cursor.close()
    conn.close()

    if not rows:
        print("No data yet.")
        return

    print(f"{'Title':<30}{'Times Borrowed':<15}")
    for row in rows:
        print(f"{row[0]:<30}{row[1]:<15}")


## 4. Try it out

Run these cells one at a time (edit the values as you like).

In [4]:
# Add a book
add_book("The Alchemist", "Paulo Coelho", "Fiction", 3)

Book 'The Alchemist' added successfully.


In [5]:
# View all books
view_books()

ID   Title                         Author              Genre          Available Total
1    The Alchemist                 Paulo Coelho        Fiction        3         3    


In [10]:
# Add a member
add_member("Jane Doe", "jane@example.com", "9876543210")

Member 'Jane Doe' added successfully.


In [11]:
# View all members
view_members()

ID   Name                     Email                         Phone          
1    Jane Doe                 jane@example.com              9876543210     


In [12]:
# Issue a book (use real book_id and member_id from the tables above)
issue_book(book_id=1, member_id=1)

Book issued successfully.


In [13]:
# View all transactions
view_transactions()

ID   Book                     Member              Issued      Returned    Status    
1    The Alchemist            Jane Doe            2026-08-06  -           issued    


In [14]:
# Return a book (use real transaction_id and book_id from the transactions table)
return_book(transaction_id=1, book_id=1)

Book returned successfully.


In [15]:
# Report: most borrowed books
most_borrowed_books()

Title                         Times Borrowed 
The Alchemist                 1              
